# ETL com Python - Dados de Leads da Internacional
Este notebook realiza o processo de ETL sobre os dados exportados do CRM da corretora Internacional.

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Adiciona caminho raiz para importações
BASE_DIR = Path.cwd().resolve().parents[1] #subiu para a raiz

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR)) # Adicionou a raiz ao caminho de busca

# Importa
from Utils.io import save_csv

# Caminho base do projeto

raw_dir = BASE_DIR / "Data" / "RAW"
processed_dir = BASE_DIR / "Data" / "PROCESSED"
final_dir = BASE_DIR / "Data" / "FINAL"
full_simulated_dir = BASE_DIR / "Data" / "RAW" / "FULL_SIMULATED"


## 1. Leitura do CSV exportado do CRM

In [2]:
# Leitura
# Caminho raiz dos arquivos simulados
base_path = Path.cwd().parents[1] / 'Data' / 'RAW' / 'FULL_SIMULATED'

# Lista todos os arquivos CSV recursivamente
csv_files = list(base_path.glob('**/*.csv'))

print(f"Total de arquivos encontrados: {len(csv_files)}")



Total de arquivos encontrados: 122


In [3]:
# Carrega todos os CSVs e concatena em um único DataFrame
df_full = pd.concat([pd.read_csv(file, parse_dates=['data_cadastro']) for file in csv_files], ignore_index=True)


In [4]:
# Verifica o total de dados consolidados
print(f"Total de linhas consolidadas: {len(df_full):,}")

Total de linhas consolidadas: 350,000


In [5]:
# Salva o DataFrame consolidado em um arquivo CSV

from datetime import datetime

# Data de hoje para versão do arquivo
hoje = datetime.now().strftime('%Y_%m_%d')
nome_arquivo = f'leads_completo_{hoje}.csv'

# Salvar o arquivo com a data no nome
save_csv(df_full, processed_dir, nome_arquivo)


✅ Arquivo salvo com sucesso: D:\pythonProject\Projeto_Dados_Corretora\Data\PROCESSED\leads_completo_2025_04_21.csv


In [6]:
df_full.head()

,lead_id,data_cadastro,origem,perfil,pais,dias_ate_1o_trade,valor_deposito,status_conversao,foi_convertido,ano,mes,semana,semana_str,target
0,49,2025-01-05,WhatsApp,Moderado,Romania,10.0,1394.3,Convertido,True,2025,1,1,1,1
1,309,2025-01-07,TikTok,Agressivo,Hong Kong,NaN,0.0,Não Convertido,False,2025,1,1,1,0
2,356,2025-01-04,Google Ads,Moderado,Norway,NaN,0.0,Não Convertido,False,2025,1,1,1,0
3,557,2025-01-07,Instagram,Moderado,Egypt,NaN,0.0,Não Convertido,False,2025,1,1,1,0
4,664,2025-01-05,Orgânico,Moderado,Burundi,NaN,0.0,Não Convertido,False,2025,1,1,1,0


## 2. Tratamento e padronização de dados

In [7]:
df = df_full.copy()
df['dias_ate_1o_trade'] = pd.to_numeric(df['dias_ate_1o_trade'], errors='coerce')
df['valor_deposito'] = pd.to_numeric(df['valor_deposito'], errors='coerce')
df['foi_convertido'] = df['status_conversao'] == 'Convertido'
df['ano_mes_cadastro'] = df['data_cadastro'].dt.to_period('M').astype(str)
df.head()

,lead_id,data_cadastro,origem,perfil,pais,dias_ate_1o_trade,valor_deposito,status_conversao,foi_convertido,ano,mes,semana,semana_str,target,ano_mes_cadastro
0,49,2025-01-05,WhatsApp,Moderado,Romania,10.0,1394.3,Convertido,True,2025,1,1,1,1,2025-01
1,309,2025-01-07,TikTok,Agressivo,Hong Kong,NaN,0.0,Não Convertido,False,2025,1,1,1,0,2025-01
2,356,2025-01-04,Google Ads,Moderado,Norway,NaN,0.0,Não Convertido,False,2025,1,1,1,0,2025-01
3,557,2025-01-07,Instagram,Moderado,Egypt,NaN,0.0,Não Convertido,False,2025,1,1,1,0,2025-01
4,664,2025-01-05,Orgânico,Moderado,Burundi,NaN,0.0,Não Convertido,False,2025,1,1,1,0,2025-01


## 3. Geração de indicadores por canal

In [8]:
agg_canais = df.groupby('origem').agg({
    'lead_id': 'count',
    'foi_convertido': 'mean',
    'valor_deposito': 'mean'
}).rename(columns={
    'lead_id': 'total_leads',
    'foi_convertido': 'taxa_conversao',
    'valor_deposito': 'deposito_medio'
}).reset_index()

agg_canais

,origem,total_leads,taxa_conversao,deposito_medio
0,E-mail Marketing,14157,0.050223,73.957917
1,Facebook,87389,0.070089,98.188992
2,Google Ads,69465,0.120708,172.644714
3,Indicação,17354,0.200818,277.572714
4,Instagram,52983,0.090236,128.504363
5,Orgânico,35248,0.099750,141.335232
6,TikTok,28015,0.079921,114.945877
7,WhatsApp,24407,0.179416,255.275072
8,YouTube Ads,20982,0.107854,153.624515


## 4. Exportação dos dados tratados para uso no Power BI ou Streamlit

In [9]:

# Salva os arquivos 
df.to_csv(processed_dir / "leads_processados_para_pbi.csv", index=False)
agg_canais.to_csv(final_dir / "indicadores_por_canal.csv", index=False)